# Análisis Exploratorio de Datos (EDA) — Detección de Deepfakes

**TFM — Máster en Ciencia de Datos e IA (UCM)**

Este notebook explora el dataset **FaceForensics++** antes de modelizar. Responde a:

1. ¿Cuántos vídeos hay y cómo se reparten entre reales y manipulados?
2. ¿Hay **desbalance de clases**? ¿Cómo afecta a la estrategia de entrenamiento?
3. ¿Qué propiedades tienen los vídeos (resolución, duración, fps)?
4. ¿Cómo se ve un fotograma real frente a uno manipulado?
5. ¿Con qué fiabilidad detecta el rostro nuestro extractor (MTCNN)?

> **Requisito:** este notebook necesita el dataset descargado en `data/raw/`
> (ver `docs/01_descarga_datos.md`). Sin los vídeos, las celdas de código no
> producirán resultados. La estructura y el código están listos para ejecutarse
> en cuanto tengas los datos.


## Preparación del entorno en Colab

La siguiente celda **solo actúa en Colab**: monta Drive, clona/actualiza el repositorio, instala dependencias y fija las rutas. **Edita `REPO_URL`** con tu repositorio. En local no hace nada y puedes saltarla.

In [ ]:
# --- Bootstrap para Google Colab (en local no hace nada) ---
import os
from pathlib import Path
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["TFM_WORKSPACE"] = "/content/drive/MyDrive/TFM_Deepfake"   # datos/figuras en Drive

    REPO_URL = "https://github.com/TU_USUARIO/TU_REPO.git"   # <-- EDITA ESTO
    PROJECT_ROOT = "/content/TFM_Deepfake_Detection"
    os.environ["TFM_PROJECT_ROOT"] = PROJECT_ROOT
    if not Path(PROJECT_ROOT).exists():
        !git clone {REPO_URL} {PROJECT_ROOT}
    else:
        !cd {PROJECT_ROOT} && git pull -q
    !pip install -q timm grad-cam gradio pyyaml tqdm
    !pip install -q --no-deps facenet-pytorch
    print("Entorno de Colab listo.")

In [ ]:
# Configuración inicial y rutas (compatible local / Google Colab)
import os
import sys
from pathlib import Path

# Detectar entorno
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # En Colab el repo se clona en una ruta conocida (ver 00_setup_colab.ipynb)
    PROJECT_ROOT = Path(os.environ.get("TFM_PROJECT_ROOT", "/content/TFM_Deepfake_Detection"))
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.seeds import set_seed, load_config
from src.utils.paths import get_paths, ensure_dirs

sns.set_theme(style="whitegrid")
cfg = load_config(PROJECT_ROOT / "config" / "config.yaml")
set_seed(cfg["project"]["seed"])

# Rutas resueltas: en Colab apuntan a Google Drive (vía TFM_WORKSPACE)
paths = get_paths(cfg, PROJECT_ROOT)
ensure_dirs(paths)
RAW = paths["raw"]
FIGURES = paths["figures"]
COMPRESSION = cfg["dataset"]["compression"]
METHODS = cfg["dataset"]["manipulation_methods"]

print("Entorno:", "Colab" if IN_COLAB else "local")
print("Raíz de datos:", RAW)
print("Figuras:", FIGURES)
print("Compresión:", COMPRESSION, "| Métodos:", METHODS)

## 1. Inventario del dataset

Construimos el inventario de vídeos: ruta, etiqueta (real=0 / fake=1), método de
manipulación y, si están disponibles, el split oficial.

In [ ]:
from src.data.dataset import enumerate_videos, load_official_splits, assign_splits

df = enumerate_videos(RAW, compression=COMPRESSION, methods=METHODS)
print(f"Total de vídeos: {len(df)}")

# Asignar split oficial si los .json están disponibles
splits_dir = RAW / "splits"
if splits_dir.exists():
    splits = load_official_splits(splits_dir)
    df["split"] = assign_splits(df, splits)
else:
    df["split"] = np.nan
    print("(Splits oficiales no encontrados en data/raw/splits — opcional)")

df.head()

## 2. Distribución de clases y desbalance

Uno de los puntos críticos del EDA: ¿cuántos reales frente a fakes? En FF++ hay muchos más vídeos manipulados (4 métodos) que originales, lo que genera un **desbalance** que debemos tratar en la modelización.

In [ ]:
# Reparto real vs fake
class_counts = df["label"].map({0: "Real", 1: "Fake"}).value_counts()
print(class_counts)
print("\nProporción de fakes: "
      f"{(df['label'] == 1).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# (a) Real vs Fake
class_counts.plot(kind="bar", ax=axes[0], color=["#2a9d8f", "#e76f51"])
axes[0].set_title("Distribución de clases (Real vs Fake)")
axes[0].set_ylabel("Nº de vídeos")
axes[0].tick_params(axis="x", rotation=0)

# (b) Por método de manipulación
method_counts = df["method"].value_counts()
method_counts.plot(kind="bar", ax=axes[1], color="#264653")
axes[1].set_title("Vídeos por categoría / método")
axes[1].set_ylabel("Nº de vídeos")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(FIGURES / "01_distribucion_clases.png", dpi=120, bbox_inches="tight")
plt.show()

> **Lectura de negocio del desbalance.** Si el modelo viera muchos más fakes que
> reales, podría "aprender" a decir *fake* por defecto y aún así parecer preciso.
> Por eso, en la modelización (Fase 2) consideraremos: balancear el muestreo,
> ponderar la función de pérdida y, sobre todo, **no fiarnos solo del *accuracy***,
> sino mirar *recall*, *precision* y la matriz de confusión.

## 3. Reparto por split oficial

Comprobamos que la partición train/val/test no introduce sesgos de clase.

In [ ]:
if df["split"].notna().any():
    tab = pd.crosstab(df["split"], df["label"].map({0: "Real", 1: "Fake"}),
                      margins=True, margins_name="Total")
    display(tab)
else:
    print("Splits no disponibles; se omite este análisis.")

## 4. Propiedades de los vídeos

Resolución, duración y fps. FF++ mezcla resoluciones (VGA, HD, Full-HD), lo que conviene conocer para el preprocesamiento. Analizamos una **muestra** para que sea rápido.

In [ ]:
from src.data.sampling import get_video_metadata

# Muestra aleatoria para no leer todos los vídeos (puede ser lento)
sample = df.sample(min(150, len(df)), random_state=cfg["project"]["seed"])

meta_rows = []
for row in sample.itertuples(index=False):
    try:
        m = get_video_metadata(row.filepath)
        m.update({"label": row.label, "method": row.method})
        meta_rows.append(m)
    except Exception as e:
        print(f"Aviso al leer {row.filepath}: {e}")

meta = pd.DataFrame(meta_rows)
meta.describe()

In [ ]:
if not meta.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Resolución (ancho x alto agrupado)
    res = (meta["width"].astype(str) + "x" + meta["height"].astype(str))
    res.value_counts().head(8).plot(kind="barh", ax=axes[0], color="#457b9d")
    axes[0].set_title("Resoluciones más frecuentes")

    axes[1].hist(meta["duration_sec"], bins=30, color="#e9c46a", edgecolor="white")
    axes[1].set_title("Duración (s)")
    axes[1].set_xlabel("segundos")

    axes[2].hist(meta["fps"], bins=20, color="#f4a261", edgecolor="white")
    axes[2].set_title("Frames por segundo (fps)")

    plt.tight_layout()
    plt.savefig(FIGURES / "02_propiedades_video.png", dpi=120, bbox_inches="tight")
    plt.show()

## 5. Visualización: fotogramas reales vs manipulados

Una inspección visual ayuda a intuir qué artefactos busca el modelo (bordes del rostro, parpadeos, texturas de piel).

In [ ]:
from src.data.sampling import sample_frames

def show_frames(video_path, title, n=4):
    frames = sample_frames(video_path, num_frames=n)
    fig, axes = plt.subplots(1, len(frames), figsize=(3 * len(frames), 3))
    if len(frames) == 1:
        axes = [axes]
    for ax, fr in zip(axes, frames):
        ax.imshow(fr)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

real_sample = df[df["label"] == 0]
fake_sample = df[df["label"] == 1]
if not real_sample.empty and not fake_sample.empty:
    show_frames(real_sample.iloc[0]["filepath"], "Ejemplo REAL")
    show_frames(fake_sample.iloc[0]["filepath"],
                f"Ejemplo FAKE ({fake_sample.iloc[0]['method']})")

## 6. Detección facial: ¿funciona MTCNN sobre estos vídeos?

Probamos el extractor en unos pocos vídeos y medimos la **tasa de detección** (qué porcentaje de frames muestreados tienen un rostro detectable). Una tasa baja sería una señal de alerta para el pipeline.

> Esta celda requiere PyTorch + facenet-pytorch instalados.

In [ ]:
try:
    from src.data.face_extraction import build_mtcnn, extract_dataset
    import tempfile

    mtcnn = build_mtcnn(
        image_size=cfg["face_extraction"]["image_size"],
        margin=cfg["face_extraction"]["margin"],
    )

    # Probar en una pequeña muestra equilibrada
    probe = pd.concat([
        df[df["label"] == 0].head(3),
        df[df["label"] == 1].head(3),
    ])

    with tempfile.TemporaryDirectory() as tmp:
        report = extract_dataset(
            probe, mtcnn, tmp,
            num_frames=cfg["face_extraction"]["frames_per_video"],
        )
    display(report)
    print(f"Tasa media de detección facial: {report['detection_rate'].mean():.1%}")
except ImportError as e:
    print(f"Instala PyTorch y facenet-pytorch para ejecutar esta celda: {e}")

## 7. Conclusiones del EDA

Resumen de hallazgos y su impacto en las siguientes fases (esta sección se
completará con los números reales tras ejecutar el notebook):

- **Volumen y clases:** el dataset es grande y está desbalanceado hacia *fakes*
  → en modelización balancearemos y priorizaremos métricas robustas al desbalance.
- **Heterogeneidad de resolución/duración** → justifica el recorte facial y el
  redimensionado a tamaño fijo (224×224).
- **Tasa de detección facial** → valida (o no) la estrategia de aislar el rostro;
  si es alta, el pipeline de preprocesamiento es fiable.
- **Coste computacional** → confirma la decisión de muestrear N frames por vídeo
  en lugar de procesar el vídeo completo.

**Siguiente paso (Fase 2):** calcular y cachear los *embeddings* de la CNN
preentrenada sobre los rostros extraídos, y entrenar el baseline a nivel de frame
y la arquitectura híbrida CNN+LSTM.
